In [0]:
%sql
DROP TABLE IF EXISTS catalog_ventas.gold.fact_ventas;

CREATE TABLE catalog_ventas.gold.fact_ventas(
    id_venta BIGINT GENERATED ALWAYS AS IDENTITY (START WITH 1 INCREMENT BY 1),
    venta BIGINT,
    id_fecha BIGINT,
    id_articulo BIGINT,
    id_cliente BIGINT,
    id_usulogin BIGINT,
    id_condvtapos BIGINT,
    delivery STRING,
    vtaestado STRING,
    precio DECIMAL(15,2),
    cant INT,
    total DECIMAL(15,2),
    _refresh_timestamp TIMESTAMP
)
USING DELTA
TBLPROPERTIES ('delta.feature.allowColumnDefaults' = 'supported')
COMMENT 'Tabla de hechos de ventas - Star Schema';


In [0]:
%sql
INSERT INTO catalog_ventas.gold.fact_ventas(
    venta,
    id_fecha,
    id_articulo,
    id_cliente,
    id_usulogin,
    id_condvtapos,
    delivery,
    vtaestado,
    precio,
    cant,
    total
)
SELECT
    v.venta,
    dc.id_fecha,
    dp.id_articulo,
    dc.id_cliente,
    dc.id_usulogin,
    dcv.id_condvtapos,
    v.delivery,
    v.vtaestado,
    v.precio,
    v.cant,
    v.total
FROM catalog_ventas.silver.ventas_clean v
LEFT JOIN catalog.gold.dim_calendario dc ON v.vtafecha = dc.vtafecha
LEFT JOIN catalog.gold.dim_producto da ON v.articulo = dp.articulo
LEFT JOIN catalog.gold.dim_clientes dc ON v.cliente = dc.cliente
LEFT JOIN catalog.gold.dim_cajeros du ON v.usulogin = dc.usulogin
LEFT JOIN catalog.gold.dim_turno dt ON v.turno = dt.turno
LEFT JOIN catalog.gold.dim_condvtapos dcv ON v.condvtapos = dcv.condvtapos

